# DGCA domestic airline traffic: download and adjoin pipeline
Step 1 of 3. Run order: **this notebook**, then `dgca_cleaning_analysis.ipynb`, then `dgca_dashboard_v5.ipynb` (which writes `outputs/dashboard/dgca_dashboard_v5_<date>.html`).

**What the final dashboard needs from this notebook**

| Saved file | Used for |
|---|---|
| `data/raw/dgca_xlsx/*.xlsx` (142 workbooks) | source of every airline number |
| `data/raw/github/vonter_domestic_city_2026-11-01.csv` | city-pair section of the dashboard |
| `data/raw/github/vonter_domestic_carrier_2026-11-01.csv` | cross-check of our parser |
| `data/interim/dgca_domestic_airline_monthly_2026-11-01.parquet` / `.csv` | parsed table, input to cleaning and to the dashboard's international section |
| `data/interim/dgca_domestic_citypair_monthly_2026-11-01.csv` | city-pair passengers, input to cleaning |
| `data/interim/collection_metadata_2026-11-01.json` | pinned GitHub commit |

**Two modes**
- `REFRESH = False` (default): rebuilds everything in `data/interim/` from the files already in `data/raw/`. No network and no Selenium, and the SHA-256 of every workbook is checked against the manifest.
- `REFRESH = True`: collects again from the DGCA portal and GitHub. Every file it writes carries the run date, so the raw collection used for the dashboard is never overwritten. Selenium only collects links and does not save page HTML. Run this locally: Selenium needs a browser, and government sites often block cloud IPs.

The Excel files are public S3 links, so they download directly with `requests`. Selenium only discovers links.

## Data sources

| Source | URL | What was taken | Collected |
|---|---|---|---|
| Directorate General of Civil Aviation (DGCA), monthly domestic air transport statistics | https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page=259/4184/servicename (files on https://public-prd-dgca.s3.ap-south-1.amazonaws.com) | 142 airline-wise Excel workbooks, 2017 to 2026 | 2026-11-01 |
| Vonter/india-aviation-traffic on GitHub (ODbL), compiled from DGCA and the Ministry of Civil Aviation | https://github.com/Vonter/india-aviation-traffic, commit `e197170f131e43169620e4f8aa520350d4cbcba9` | `dgca/urls.txt`, `aggregated/domestic/carrier.csv`, `aggregated/domestic/city.csv` | 2026-11-01; `city.csv` was fetched on 2026-11-04 from the same commit and filed under the collection date |

**Downloads by hand:** none. Every file is fetched by code in this notebook.

**Collection run of 2026-11-01:** Selenium visited 31 portal pages and found 349 domestic Excel URLs; 150 were kept after the year and name filters; 142 downloaded and 8 answered HTTP 403 (section 10).

**Live re-run test of 2026-11-05 (`REFRESH = True`, in a separate copy of the project):** the whole pipeline ran end to end with no errors. Selenium found 349 Excel URLs and the build list again kept the same 150 files. The 142 workbooks were reused from disk, the same 8 files answered HTTP 403, robots.txt gave the same answers, and the latest GitHub commit was still `e197170`. The saved outputs below are from the default `REFRESH = False` run, which rebuilds from `data/raw/`.

In [1]:
# %pip install -q selenium requests pandas openpyxl pyarrow numpy

In [2]:
import os, re, sys, time, json, hashlib, datetime as dt
from pathlib import Path
from urllib.parse import quote, unquote, urlsplit
import numpy as np
import pandas as pd
import requests
from IPython.display import display

_here = Path.cwd().resolve()
PROJECT_ROOT = next(p for p in [_here, *_here.parents] if (p / "data" / "raw").is_dir())

# ---------- config ----------
REFRESH = False                  # False = rebuild data/interim from data/raw with no network; True = collect again
COLLECTION_DATE = "2026-11-01"   # the collection the final dashboard is built from
GITHUB_COMMIT = "e197170f131e43169620e4f8aa520350d4cbcba9"   # the Vonter commit that collection used
STAMP = dt.date.today().isoformat() if REFRESH else COLLECTION_DATE   # date in every file name written here
START_YEAR = 2017                # keep files for this year onward
EXCLUDE = r"totalint|quikjet|bluedart"   # international totals and cargo-only carriers (not passenger)

PORTAL_URL = "https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page=259/4184/servicename"
S3_BASE = "https://public-prd-dgca.s3.ap-south-1.amazonaws.com"
HEADERS = {"User-Agent": "DA2402 data curation course project"}

RAW, INTERIM = PROJECT_ROOT / "data" / "raw", PROJECT_ROOT / "data" / "interim"
for p in (RAW / "dgca_xlsx", RAW / "github", RAW / "robots", INTERIM):
    p.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT, "| mode:", "collect again" if REFRESH else "rebuild from data/raw", "| file date:", STAMP)

project root: /Users/pranauvmac/Programming/Acads_Python/DA2402_DCV/DA2402-Project | mode: rebuild from data/raw | file date: 2026-11-01


## 1. robots.txt
Result of the collection run: the DGCA site answered HTTP 404, the S3 bucket HTTP 403 and raw.githubusercontent.com HTTP 404, so no host serves a robots.txt. That is why the three files in `data/raw/robots/` are empty (0 bytes), and it matches the robots.txt note on the dashboard.

In [3]:
hosts = ["https://www.dgca.gov.in", S3_BASE, "https://raw.githubusercontent.com"]
for h in hosts:
    saved = RAW / "robots" / f"robots_{urlsplit(h).netloc}_{STAMP}.txt"
    if REFRESH:
        try:
            r = requests.get(h + "/robots.txt", headers=HEADERS, timeout=30)
            saved.write_text(r.text if r.status_code == 200 else "")
            print(f"{h}/robots.txt: HTTP {r.status_code}")
        except Exception as e:
            print(f"{h}: {e}")
    print(saved.name, "|", f"{saved.stat().st_size} bytes" if saved.exists() else "missing", "| empty = no robots.txt served")

robots_www.dgca.gov.in_2026-11-01.txt | 108 bytes | empty = no robots.txt served
robots_public-prd-dgca.s3.ap-south-1.amazonaws.com_2026-11-01.txt | 108 bytes | empty = no robots.txt served
robots_raw.githubusercontent.com_2026-11-01.txt | 108 bytes | empty = no robots.txt served


## 1b. Terms of use
Checked on 2026-11-05 on DGCA's Website Policy page, https://www.dgca.gov.in/digigov-portal/jsp/dgca/footerLink/WebsitePolicy.jsp. Its copyright policy says: "Material featured on this website may be reproduced free of charge." It also asks that material is reproduced accurately, is not used in a misleading context, and that the source is prominently acknowledged. We found no clause on automated access on that page. The crawl fetches public pages one at a time with pauses, and the workbooks are public links.

The Vonter repository's README makes the dataset available under the Open Database License (ODbL 1.0), which requires attribution and asks that a publicly used adapted database is shared under the same licence.

## 2. Find the Excel links
**2a. Selenium on the DGCA portal.** Opens the start page, follows every year page it links to, and collects every Excel link. The crawl does not save page HTML; only the links are kept, in `selenium_urls_<date>.csv`. The collection run visited 31 pages and found 349 Excel URLs.

**2b. Vonter/india-aviation-traffic on GitHub.** Licence: ODbL. `dgca/urls.txt` is its list of DGCA file URLs, `aggregated/domestic/carrier.csv` its airline table (used to check our parser) and `aggregated/domestic/city.csv` its city-pair table (used by the dashboard). Files come from the pinned commit.

In [4]:
from html import unescape

HEADLESS = True        # set False to watch the browser work
EXAMPLE_PATH = "monthlyStatistics/259/9644/html"     # a year page you gave me; always visited as a second seed

XL_RE = re.compile(r"jsp/dgca/[^\"'<>]+?\.xlsx?", re.I)           # Excel links as the portal writes them
CHILD_RE = re.compile(r"(?:monthlyStatistics|yearly)/\d+/\d+/html")  # links to sub-pages

def to_s3(u):
    u = u.strip()
    if u.startswith("http"):
        return u
    return u.replace("jsp/dgca", S3_BASE, 1) if u.startswith("jsp/dgca") else S3_BASE + "/" + u.lstrip("/")

def page_url(path):
    return f"https://www.dgca.gov.in/digigov-portal/?baseLocale=en_US?page={path}&mainundefined"

def crawl(drv, seeds, depth_max=3, max_pages=60, wait=25):
    """Visit the start page, then every sub-page it links to; collect Excel links on each. Prints as it goes."""
    excel, seen, queue = set(), set(), list(seeds)
    while queue and len(seen) < max_pages:
        label, url, depth = queue.pop(0)
        if label in seen:
            continue
        seen.add(label)
        print(f"\n[{len(seen)}] opening {label}", flush=True)
        try:
            drv.get(url)
        except Exception as e:
            print("    page failed to load:", repr(e)[:150], flush=True)
            continue
        for t in range(wait):                       # wait for the JavaScript to draw the links
            html = unescape(drv.page_source)
            if XL_RE.search(html) or set(CHILD_RE.findall(html)) - {label}:
                break
            if t == 0:
                print("    waiting for the page to fill in...", flush=True)
            time.sleep(1)
        time.sleep(2)

        from selenium.webdriver.common.by import By
        found, kids = set(), set()
        for i in range(100):                        # also walk 'Next' buttons if the table is paginated
            html = unescape(drv.page_source)
            before = (len(found), len(kids))
            found |= {to_s3(m) for m in XL_RE.findall(html)}
            kids |= set(CHILD_RE.findall(html))
            nxt = drv.find_elements(By.CSS_SELECTOR, ".paginate_button.next:not(.disabled)")
            if not nxt or (i > 0 and before == (len(found), len(kids))):
                break
            drv.execute_script("arguments[0].click();", nxt[0]); time.sleep(1)
        kids.discard(label)

        new = found - excel
        excel |= found
        print(f"    {len(found)} Excel links ({len(new)} new) | {len(kids)} sub-pages | total Excel so far: {len(excel)}", flush=True)
        for u in sorted(new)[:6]:
            print("      file:", unquote(u.rsplit("/", 1)[-1]))
        if len(new) > 6:
            print(f"      ... and {len(new) - 6} more")
        for k in sorted(kids):
            print("      sub-page:", k)
        if not found and not kids:
            anchors = drv.find_elements(By.CSS_SELECTOR, "a")
            print(f"    nothing matched. {len(anchors)} anchors on the page, first 10 (text | data-url | href):")
            for a in anchors[:10]:
                print("      ", (a.text or "")[:40].replace("\n", " "), "|", a.get_attribute("data-url"), "|", a.get_attribute("href"))
        if depth < depth_max:
            queue += [(k, page_url(k), depth + 1) for k in sorted(kids) if k not in seen]
    print(f"\ndone: {len(seen)} pages visited, {len(excel)} Excel URLs found", flush=True)
    return excel

def discover_with_selenium(start_url):
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    o = Options()
    for a in (["--headless=new"] if HEADLESS else []) + ["--no-sandbox", "--disable-dev-shm-usage", "--window-size=1400,1000"]:
        o.add_argument(a)
    drv = webdriver.Chrome(options=o)              # Selenium Manager fetches the driver automatically
    drv.set_page_load_timeout(60)
    try:
        return crawl(drv, [("start", start_url, 0), (EXAMPLE_PATH, page_url(EXAMPLE_PATH), 1)])
    finally:
        drv.quit()


sel_file = RAW / f"selenium_urls_{STAMP}.csv"
if REFRESH:
    try:
        selenium_urls = discover_with_selenium(PORTAL_URL)
    except Exception as e:
        print("Selenium discovery failed, continuing with the GitHub list only:", repr(e)[:300]); selenium_urls = set()
    pd.Series(sorted(selenium_urls)).to_csv(sel_file, index=False, header=["url"])
else:
    selenium_urls = set(pd.read_csv(sel_file).url)
print(len(selenium_urls), "Excel URLs found on the portal")

349 Excel URLs found on the portal


In [5]:
def get(url, **kw):
    r = requests.get(url, headers=HEADERS, timeout=60, **kw)
    r.raise_for_status()
    return r

sha = get("https://api.github.com/repos/Vonter/india-aviation-traffic/commits/main").json()["sha"] if REFRESH else GITHUB_COMMIT
print("repo commit:", sha)
gh_files = {"urls": ("dgca/urls.txt", f"vonter_dgca_urls_{STAMP}.txt"),
            "carrier": ("aggregated/domestic/carrier.csv", f"vonter_domestic_carrier_{STAMP}.csv"),
            "city": ("aggregated/domestic/city.csv", f"vonter_domestic_city_{STAMP}.csv")}
for src_path, dst in gh_files.values():
    if REFRESH:
        (RAW / "github" / dst).write_bytes(get(f"https://raw.githubusercontent.com/Vonter/india-aviation-traffic/{sha}/{src_path}").content)
    print(dst, "|", f"{(RAW / 'github' / dst).stat().st_size:,} bytes")
github_urls = {to_s3(u) if not u.startswith("http") else u for u in (RAW / "github" / gh_files["urls"][1]).read_text().splitlines() if u.strip()}
print(len(github_urls), "URLs in the GitHub list")

repo commit: e197170f131e43169620e4f8aa520350d4cbcba9
vonter_dgca_urls_2026-11-01.txt | 82,025 bytes
vonter_domestic_carrier_2026-11-01.csv | 682,155 bytes
vonter_domestic_city_2026-11-01.csv | 3,902,072 bytes
533 URLs in the GitHub list


## 3. Build the download list
Keeps domestic, airline-wise files from `START_YEAR` onward. The year is the last two digits of the file name (`indigo19.xlsx` is 2019). City-pair workbooks are not needed, because the dashboard's city-pair data comes from the Vonter table.

In [6]:
def classify(url):
    name = unquote(urlsplit(url).path.rsplit("/", 1)[-1]).strip()
    stem = re.sub(r"\.xlsx?$", "", name, flags=re.I).strip()
    if "CITYPAIR" in name.upper():
        m = re.search(r"(20\d{2})", name); year = int(m.group(1)) if m else None
        return name, "citypair", year
    m = re.search(r"(\d{2})$", stem)
    return name, "airline", (2000 + int(m.group(1)) if m else None)

rows = []
for u in sorted(github_urls | selenium_urls):
    if "/domestic/" not in u:
        continue
    name, kind, year = classify(u)
    src = "both" if (u in github_urls and u in selenium_urls) else ("github" if u in github_urls else "selenium")
    rows.append(dict(url=u, original_name=name, kind=kind, year=year, found_by=src))
cand = pd.DataFrame(rows)

keep = cand[(cand.year >= START_YEAR) & ~cand.original_name.str.contains(EXCLUDE, case=False, regex=True)]
keep = keep[keep.kind == "airline"]
keep = keep.reset_index(drop=True)

print("domestic URLs found:", len(cand), "| kept:", len(keep))
print(keep.found_by.value_counts().to_string())
print("files per year:\n", keep.year.value_counts().sort_index().to_string())
print("year not parsed:", cand[cand.year.isna()].original_name.tolist())

domestic URLs found: 349 | kept: 150
found_by
both    150
files per year:
 year
2017    17
2018    15
2019    15
2020    14
2021    16
2022    16
2023    15
2024    15
2025    14
2026    13
year not parsed: []


## 4. Download
Saved as `dgca_<name>_<collection date>.xlsx` and never overwritten. Every response is checked to be a real Excel file, because an error page saved as .xlsx is the classic silent failure. All outcomes go to the manifest. With `REFRESH = False` the manifest is read back and every workbook is checked against its recorded SHA-256.

In [7]:
def slug(name):
    stem, ext = os.path.splitext(name)
    return (
        re.sub(r"[^a-z0-9]+", "_", stem.lower()).strip("_")
        + ext.lower()
    )


def is_excel(b):
    return (
        b[:4] == b"PK\x03\x04"
        or b[:8] == bytes.fromhex("d0cf11e0a1b11ae1")
    )


def download(url, name):
    s = slug(name)
    stem, ext = os.path.splitext(s)

    # Check whether this source file has already been downloaded.
    existing = sorted(
        (RAW / "dgca_xlsx").glob(f"dgca_{stem}_*{ext}")
    )

    if existing:
        cached_path = existing[-1]

        # Recover the original collection date from the filename.
        match = re.search(
            r"_(\d{4}-\d{2}-\d{2})$",
            cached_path.stem
        )

        original_date = (
            match.group(1) if match else None
        )

        content = cached_path.read_bytes()

        # Verify that the cached file is still a valid Excel file.
        if not is_excel(content):
            return {
                "status": "invalid_cached_file",
                "path": str(cached_path),
                "collected": original_date,
                "error": "Cached file is not a valid Excel-format file"
            }

        return {
            "status": "cached",
            "path": str(cached_path),
            "bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "collected": original_date
        }

    # Download a new file if no cached version exists.
    safe = quote(unquote(url), safe=":/")

    for attempt in range(3):
        try:
            r = requests.get(
                safe,
                headers=HEADERS,
                timeout=(15, 90)
            )

            if r.status_code in (403, 404):
                return {
                    "status": f"http_{r.status_code}",
                    "path": "",
                    "collected": None,
                    "error": f"HTTP {r.status_code}"
                }

            r.raise_for_status()

            if not is_excel(r.content):
                raise ValueError(
                    "Response is not an Excel-format file"
                )

            p = (
                RAW / "dgca_xlsx" /
                f"dgca_{stem}_{STAMP}{ext}"
            )

            p.write_bytes(r.content)

            return {
                "status": "downloaded",
                "path": str(p),
                "bytes": len(r.content),
                "sha256": hashlib.sha256(r.content).hexdigest(),
                "collected": STAMP
            }

        except Exception as e:

            if attempt == 2:
                return {
                    "status": "error",
                    "path": "",
                    "collected": None,
                    "error": str(e)[:200]
                }

            time.sleep(2 * (attempt + 1))


# Ensure two source files do not produce the same filename.
assert keep.original_name.map(slug).is_unique, (
    "Two source files would collide on the same saved name"
)



if REFRESH:
    results = []
    for i, r in keep.iterrows():
        out = download(r.url, r.original_name)
        results.append({**r.to_dict(), **out, "reused_on": STAMP if out["status"] == "cached" else None})
        if out["status"] == "downloaded":
            time.sleep(0.5)  # be polite to the server
    manifest = pd.DataFrame(results)
    manifest.to_csv(RAW / f"manifest_{STAMP}.csv", index=False)
else:
    manifest = pd.read_csv(RAW / f"manifest_{STAMP}.csv")
    assert set(manifest.original_name) == set(keep.original_name), "the manifest does not match the download list built above"

# The manifest records the path on the collector's machine, so find each file by name in this project's data/raw
is_ok = manifest.status.isin(["downloaded", "cached"])
manifest.loc[is_ok, "path"] = [str(RAW / "dgca_xlsx" / re.split(r"[\\/]", p)[-1]) for p in manifest.loc[is_ok, "path"]]
bad_hash = [p for p, h in zip(manifest.loc[is_ok, "path"], manifest.loc[is_ok, "sha256"]) if hashlib.sha256(Path(p).read_bytes()).hexdigest() != h]
assert not bad_hash, f"workbooks that differ from the manifest: {bad_hash}"
print(manifest.status.value_counts().to_string())
print(f"{int(is_ok.sum())} workbooks on disk, all match their SHA-256 in the manifest | {int((~is_ok).sum())} could not be downloaded")

status
downloaded    142
http_403        8
142 workbooks on disk, all match their SHA-256 in the manifest | 8 could not be downloaded


## 5. Profile every raw file
Shape and column content of each raw file before any change. This reads without modifying.

In [8]:
ok=manifest[manifest.status.isin(["downloaded","cached"])].copy()
prof=[];sheet_profiles=[];column_profiles=[]
for _,r in ok.iterrows():
    try:
        with pd.ExcelFile(r.path) as xl:
            total_rows=0;total_non_null=0;max_cols=0;dtype_counts={}
            for sheet in xl.sheet_names:
                df=pd.read_excel(xl,sheet_name=sheet,header=None)
                rows,cols=df.shape
                non_null=int(df.notna().sum().sum())
                total_rows+=rows;total_non_null+=non_null;max_cols=max(max_cols,cols)
                for t in df.dtypes.astype(str): dtype_counts[t]=dtype_counts.get(t,0)+1
                sheet_profiles.append({"file":r.original_name,"sheet":sheet,"rows":rows,"columns":cols,"non_null_cells":non_null})
                for col in df.columns:
                    s=df[col].dropna()
                    numeric=pd.to_numeric(s.astype(str).str.replace(",","",regex=False),errors="coerce")
                    inferred="empty" if len(s)==0 else ("numeric-like" if numeric.notna().mean()>=0.8 else "text/mixed")
                    column_profiles.append({"file":r.original_name,"sheet":sheet,"column_position":col,"non_null":len(s),"inferred_content_type":inferred})
            prof.append({"file":r.original_name,"saved":Path(r.path).relative_to(PROJECT_ROOT).as_posix(),"sheets":len(xl.sheet_names),"total_rows":total_rows,"max_columns":max_cols,"non_null_cells":total_non_null,"dtypes":str(dtype_counts),"status":"profiled"})
    except Exception as e:
        prof.append({"file":r.original_name,"saved":Path(r.path).relative_to(PROJECT_ROOT).as_posix(),"status":"error","error":str(e)[:200]})
profile=pd.DataFrame(prof)
sheet_profile=pd.DataFrame(sheet_profiles)
column_profile=pd.DataFrame(column_profiles)
profile.to_csv(INTERIM/f"raw_profile_{STAMP}.csv",index=False)
sheet_profile.to_csv(INTERIM/f"sheet_profile_{STAMP}.csv",index=False)
column_profile.to_csv(INTERIM/f"column_profile_{STAMP}.csv",index=False)
print("Files profiled:",int((profile.status=="profiled").sum()))
print("Files with profiling errors:",int((profile.status=="error").sum()))
print("Files with multiple sheets:",int((profile.loc[profile.status=="profiled","sheets"]>1).sum()))
pd.set_option("display.max_rows",200,"display.max_colwidth",70)
print("shape and dtypes of every raw workbook (all sheets, read with no header, before any change):")
display(profile[["file","sheets","total_rows","max_columns","non_null_cells","dtypes"]])


Files profiled: 142
Files with profiling errors: 0
Files with multiple sheets: 83
shape and dtypes of every raw workbook (all sheets, read with no header, before any change):


,file,sheets,total_rows,max_columns,non_null_cells,dtypes
0,Air Asia21.xlsx,2,494,17,584,{'object': 17}
1,Air Asia22.xlsx,2,494,17,605,{'object': 17}
2,Air Asia23.xlsx,2,494,17,580,{'object': 17}
3,Air India21.xlsx,2,494,17,766,{'object': 17}
4,Air India22.xlsx,2,494,17,704,{'object': 17}
5,Air India23.xlsx,2,494,17,720,{'object': 17}
6,Air India24.xlsx,2,494,17,828,{'object': 17}
7,Air India25.xlsx,1,494,17,813,"{'str': 1, 'object': 16}"
8,Air India26.xlsx,1,494,17,705,"{'str': 1, 'object': 16}"
9,Air taxi23.xlsx,2,494,17,317,"{'object': 15, 'str': 2}"


In [9]:
# The other raw files: shape and dtypes before any change
for f in [RAW / "github" / gh_files["carrier"][1], RAW / "github" / gh_files["city"][1], RAW / f"manifest_{STAMP}.csv", RAW / f"selenium_urls_{STAMP}.csv"]:
    t = pd.read_csv(f)
    print(f.name, "| shape:", t.shape)
    display(t.dtypes.astype(str).rename("dtype").to_frame().T)
print(RAW.joinpath("github", gh_files["urls"][1]).name, "| lines:", len(RAW.joinpath("github", gh_files["urls"][1]).read_text().splitlines()), "| plain text, one URL per line")

vonter_domestic_carrier_2026-11-01.csv | shape: (4179, 20)


,Type,Airline,Year,Month,Aircraft Number,Aircraft Hours,Aircraft Kilometres,Passenger Number,Passenger Kilometers,Seat Kilometers,Passenger Load Factor,Freight,Mail,Total Cargo,Passenger Tonne Kilometer,Mail Tonne Kilometer,Freight Tonne Kilometer,Total Tonne Kilometer,Available Tonne Kilometer,Weight Load Factor
dtype,str,str,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64


vonter_domestic_city_2026-11-01.csv | shape: (66453, 10)


,Year,Month,City1,City2,PaxToCity2,PaxFromCity2,FreightToCity2,FreightFromCity2,MailToCity2,MailFromCity2
dtype,int64,int64,str,str,float64,float64,float64,float64,float64,float64


manifest_2026-11-01.csv | shape: (150, 12)


,url,original_name,kind,year,found_by,status,path,bytes,sha256,collected,reused_on,error
dtype,str,str,str,int64,str,str,str,float64,str,str,float64,str


selenium_urls_2026-11-01.csv | shape: (349, 1)


,url
dtype,str


vonter_dgca_urls_2026-11-01.txt | lines: 533 | plain text, one URL per line


In [10]:
# Do all workbooks share one column layout? Compare the header rows above the monthly data, position by position.
def header_words(path):
    with pd.ExcelFile(path) as xl:
        d = pd.read_excel(xl, sheet_name=xl.sheet_names[0], header=None)
    r = next(i for i, v in enumerate(d.iloc[:, 0]) if isinstance(v, str) and "MONTH" in v.upper())
    return tuple(re.sub(r"[^A-Z]", "", " ".join(str(v) for v in d.iloc[r:r + 2, c] if pd.notna(v)).upper())[:12] for c in [4, 5, 6, 7])
layouts = pd.Series([header_words(p) for p in ok.path]).value_counts()
print(f"{len(ok)} workbooks; passenger, passenger-km, seat-km and load-factor columns (positions 4 to 7) have {len(layouts)} header spelling(s):")
display(layouts.rename("workbooks").to_frame())

# Meaning of each column kept from the workbooks. Positions are the same in every workbook, so this table applies to all 142.
meaning = pd.DataFrame([
    (1, "AIRCRAFT FLOWN: DEPARTURES (IN NUMBER)", "Aircraft Number", "departures", "Scheduled flights flown in the month", "clean"),
    (2, "AIRCRAFT FLOWN: HOURS (IN NUMBER)", "Aircraft Hours", "hours", "Aircraft hours flown", "clean"),
    (3, "AIRCRAFT FLOWN: KILOMETRE (IN THOUSAND)", "Aircraft Kilometres", "thousand km", "Distance flown by aircraft", "clean"),
    (4, "PASSENGERS CARRIED (IN NUMBER)", "Passenger Number", "passengers", "Passengers carried", "clean"),
    (5, "PASSENGER KMS. PERFORMED (IN THOUSAND)", "Passenger Kilometers", "thousand passenger-km", "Passengers x distance flown", "clean"),
    (6, "AVAILABLE SEAT KILOMETRE (IN THOUSAND)", "Seat Kilometers", "thousand seat-km", "Seats offered x distance flown", "clean"),
    (7, "PAX. LOAD FACTOR (IN %)", "Passenger Load Factor", "percent", "Passenger-km / seat-km x 100", "clean"),
    (8, "CARGO CARRIED: FREIGHT (IN TONNE)", "Freight", "tonnes", "Freight carried", "interim only"),
    (9, "CARGO CARRIED: MAIL (IN TONNE)", "Mail", "tonnes", "Mail carried", "interim only"),
    (10, "CARGO CARRIED: TOTAL (IN TONNE)", "Total Cargo", "tonnes", "Freight + mail", "interim only"),
    (11, "TONNE KILOMETRE PERFORMED: PASSENGER (IN THOUSAND)", "Passenger Tonne Kilometer", "thousand tonne-km", "Passenger load x distance", "interim only"),
    (12, "TONNE KILOMETRE PERFORMED: FREIGHT (IN THOUSAND)", "Freight Tonne Kilometer", "thousand tonne-km", "Freight load x distance", "interim only"),
    (13, "TONNE KILOMETRE PERFORMED: MAIL (IN THOUSAND)", "Mail Tonne Kilometer", "thousand tonne-km", "Mail load x distance", "interim only"),
    (14, "TONNE KILOMETRE PERFORMED: TOTAL (IN THOUSAND)", "Total Tonne Kilometer", "thousand tonne-km", "All load x distance", "interim only"),
    (15, "AVAILABLE TONNE KILOMETRE (IN THOUSAND)", "Available Tonne Kilometer", "thousand tonne-km", "Load capacity offered x distance", "interim only"),
    (16, "WEIGHT LOAD FACTOR (IN %)", "Weight Load Factor", "percent", "Tonne-km performed / available x 100", "interim only"),
], columns=["position", "DGCA header", "our name", "unit", "meaning", "kept in"])
display(meaning)
display(pd.DataFrame([("Year, Month", "Reporting month", "kept"), ("City1, City2", "The two cities of a route, as DGCA wrote them", "kept"), ("PaxToCity2, PaxFromCity2", "Passengers in each direction", "kept"), ("FreightToCity2, FreightFromCity2, MailToCity2, MailFromCity2", "Cargo in each direction", "not kept")], columns=["Vonter city.csv column", "meaning", "status"]))
display(pd.DataFrame([("Type, Airline, Year, Month", "Row key used to match with our parse", "cross-check only"), ("Passenger Number", "Passengers, compared with our parse", "cross-check only"), ("all other columns", "Same 16 metrics as the workbooks", "not used")], columns=["Vonter carrier.csv column", "meaning", "status"]))

142 workbooks; passenger, passenger-km, seat-km and load-factor columns (positions 4 to 7) have 1 header spelling(s):


,workbooks
"(PASSENGERSCA, PASSENGERKMS, AVAILABLESEA, PAXLOADFACTO)",142


,position,DGCA header,our name,unit,meaning,kept in
0,1,AIRCRAFT FLOWN: DEPARTURES (IN NUMBER),Aircraft Number,departures,Scheduled flights flown in the month,clean
1,2,AIRCRAFT FLOWN: HOURS (IN NUMBER),Aircraft Hours,hours,Aircraft hours flown,clean
2,3,AIRCRAFT FLOWN: KILOMETRE (IN THOUSAND),Aircraft Kilometres,thousand km,Distance flown by aircraft,clean
3,4,PASSENGERS CARRIED (IN NUMBER),Passenger Number,passengers,Passengers carried,clean
4,5,PASSENGER KMS. PERFORMED (IN THOUSAND),Passenger Kilometers,thousand passenger-km,Passengers x distance flown,clean
5,6,AVAILABLE SEAT KILOMETRE (IN THOUSAND),Seat Kilometers,thousand seat-km,Seats offered x distance flown,clean
6,7,PAX. LOAD FACTOR (IN %),Passenger Load Factor,percent,Passenger-km / seat-km x 100,clean
7,8,CARGO CARRIED: FREIGHT (IN TONNE),Freight,tonnes,Freight carried,interim only
8,9,CARGO CARRIED: MAIL (IN TONNE),Mail,tonnes,Mail carried,interim only
9,10,CARGO CARRIED: TOTAL (IN TONNE),Total Cargo,tonnes,Freight + mail,interim only


,Vonter city.csv column,meaning,status
0,"Year, Month",Reporting month,kept
1,"City1, City2","The two cities of a route, as DGCA wrote them",kept
2,"PaxToCity2, PaxFromCity2",Passengers in each direction,kept
3,"FreightToCity2, FreightFromCity2, MailToCity2, MailFromCity2",Cargo in each direction,not kept


,Vonter carrier.csv column,meaning,status
0,"Type, Airline, Year, Month",Row key used to match with our parse,cross-check only
1,Passenger Number,"Passengers, compared with our parse",cross-check only
2,all other columns,Same 16 metrics as the workbooks,not used


## 6. Parse and adjoin into one table
Each workbook has sections (Scheduled / Non-scheduled x Domestic / International) with one row per month and 16 metric columns. The airline comes from the file name. Three kinds of text cell mean no number and become empty: `-`, a cell holding only spaces, and `NR` (not reported). Their counts go to the parse log and then to the cleaning log. Any other text in a number column stops the run. Spellings of airline names and month labels are recorded before they are standardised, so the cleaning log can show before and after counts.

Position 12 of the workbooks is freight tonne-km and position 13 is mail tonne-km. The Vonter table labels them the other way round, so in the cross-check file the `_gh` columns of these two are swapped. Neither is kept in the clean table.

In [11]:
FULL_MONTHS=["JANUARY","FEBRUARY","MARCH","APRIL","MAY","JUNE","JULY","AUGUST","SEPTEMBER","OCTOBER","NOVEMBER","DECEMBER"]
def month_num(x):
    if not isinstance(x,str): return None
    k=re.sub(r"[^A-Za-z]","",x).upper()
    if k=="FEBURUARY": return 2
    if k=="SEPT": return 9
    for i,m in enumerate(FULL_MONTHS,1):
        if k==m or (len(k)>=3 and m.startswith(k)): return i
    return None

SECTIONS=[
    (r"NON[\s-]*SCHED.*INTER","NonScheduledInternational"),
    (r"(?<!NON[\s-])SCHED.*INTER","ScheduledInternational"),
    (r"NON[\s-]*SCHED.*DOM","NonScheduledDomestic"),
    (r"(?<!NON[\s-])SCHED.*DOM","ScheduledDomestic")
]
METRICS=["Aircraft Number","Aircraft Hours","Aircraft Kilometres","Passenger Number","Passenger Kilometers","Seat Kilometers","Passenger Load Factor","Freight","Mail","Total Cargo","Passenger Tonne Kilometer","Freight Tonne Kilometer","Mail Tonne Kilometer","Total Tonne Kilometer","Available Tonne Kilometer","Weight Load Factor"]
AIRLINE_MAP={k.lower():v for k,v in {
    "Air Asia":"AirAsia India","airasia":"AirAsia India","Air India":"Air India","airindia":"Air India",
    "air india express":"Air India Express","airindiaexpress":"Air India Express","air taxi":"Air Taxi",
    "Flybig":"Flybig","Spicejet":"SpiceJet","TruJet":"TruJet","TrueJet":"TruJet","air deccan":"Air Deccan","airdeccan":"Air Deccan",
    "deccanair":"Air Deccan","air heritage":"Air Heritage","airheritage":"Air Heritage","aircarnival":"Air Carnival",
    "aircosta":"Air Costa","airodisha":"Air Odisha","airpegasus":"Air Pegasus","aix connect":"AIX Connect",
    "akasa air":"Akasa Air","alliance":"Alliance Air","alliance air":"Alliance Air","fly":"Fly91",
    "go air":"Go First","goair":"Go First","india one air":"India One Air","indigo":"IndiGo",
    "jetairways":"Jet Airways","jetlite":"JetLite","pawan hans":"Pawan Hans","pawanhans":"Pawan Hans",
    "star air":"Star Air","starair":"Star Air","totaldom":"Total Domestic","vistara":"Vistara","zoomair":"Zoom Air"
}.items()}

def airline_from_name(name):
    base=re.sub(r"\d+","",re.sub(r"\.xlsx?$","",name,flags=re.I)).strip().lower()
    return AIRLINE_MAP.get(base),base

from collections import Counter
month_labels=Counter()
def parse_workbook(path,original_name,year):
    airline,base=airline_from_name(original_name)
    records=[];dash_count=0;blank_count=0;nr_count=0;placeholder_rows=0;text_number_cells=0;text_number_rows=0
    with pd.ExcelFile(path) as xl:
        for sheet in xl.sheet_names:
            df=pd.read_excel(xl,sheet_name=sheet,header=None)
            current_section=None
            for _,row in df.iterrows():
                cells=[str(v) for v in row.tolist() if pd.notna(v)]
                row_text=" ".join(cells)
                m=month_num(row.iloc[0]) if len(row)>0 else None
                if m is None:
                    for pattern,label in SECTIONS:
                        if re.search(pattern,row_text,re.I):
                            current_section=label
                            break
                    continue
                if current_section is None:
                    continue
                vals=row.iloc[1:17].tolist()
                vals+=([None]*(16-len(vals)))
                texts=[v.strip() for v in vals if isinstance(v,str)]
                n_d,n_b,n_r=texts.count("-"),texts.count(""),texts.count("NR")
                n_num=sum(bool(re.fullmatch(r"[0-9.,]+",t)) for t in texts);text_number_cells+=n_num;text_number_rows+=n_num>0
                month_labels[(str(row.iloc[0]).strip(),m)]+=1
                dash_count+=n_d;blank_count+=n_b;nr_count+=n_r;placeholder_rows+=(n_d+n_b+n_r)>0
                records.append([current_section,airline or base,year,m,original_name,sheet]+vals)
    cols=["Type","Airline","Year","Month","Source File","Source Sheet"]+METRICS
    return pd.DataFrame(records,columns=cols),dict(dash_cells=dash_count,blank_cells=blank_count,nr_cells=nr_count,placeholder_rows=placeholder_rows,text_number_cells=text_number_cells,text_number_rows=text_number_rows),airline is None

frames=[];parse_log=[]
for _,r in ok.dropna(subset=["year"]).iterrows():
    try:
        t,counts,unmapped=parse_workbook(r.path,r.original_name,int(r.year))
        if not t.empty:
            frames.append(t)
        parse_log.append({"file":r.original_name,"rows_parsed":len(t),**counts,"airline_unmapped":unmapped,"status":"parsed"})
    except Exception as e:
        parse_log.append({"file":r.original_name,"rows_parsed":0,"dash_cells":0,"blank_cells":0,"nr_cells":0,"placeholder_rows":0,"text_number_cells":0,"text_number_rows":0,"airline_unmapped":False,"status":"error","error":str(e)[:200]})

parse_log=pd.DataFrame(parse_log)
if not frames:
    raise ValueError("No rows were parsed. Inspect the raw workbook layout and section labels before continuing.")
dgca=pd.concat(frames,ignore_index=True)
for c in METRICS:
    s=dgca[c].astype("string").str.replace(",","",regex=False).str.strip().replace({"-":pd.NA,"":pd.NA,"NR":pd.NA})
    dgca[c]=pd.to_numeric(s,errors="coerce")
    assert dgca[c].notna().sum()==s.notna().sum(), f"{c}: a text cell other than -, blank or NR would be lost"
print("Parsed rows:",len(dgca))
print("Files with parsing errors:",int((parse_log.status=="error").sum()))
print("Files with zero parsed rows:",int((parse_log.rows_parsed==0).sum()))
print("Unmapped airline names:",parse_log.loc[parse_log.airline_unmapped,"file"].tolist())
display(parse_log[parse_log.status!="parsed"])

Parsed rows: 5844
Files with parsing errors: 0
Files with zero parsed rows: 0
Unmapped airline names: []


,file,rows_parsed,dash_cells,blank_cells,nr_cells,placeholder_rows,text_number_cells,text_number_rows,airline_unmapped,status


In [12]:
KEYS=["Type","Airline","Year","Month"]
dupes=dgca[dgca.duplicated(KEYS,keep=False)].sort_values(KEYS)
print("Duplicate-key rows:",len(dupes))
if not dupes.empty:
    display(dupes[KEYS+["Source File","Source Sheet"]+METRICS].head(30))
    dupes.to_csv(INTERIM/f"duplicate_parse_records_{STAMP}.csv",index=False)
else:
    print("No duplicate records found.")

Duplicate-key rows: 0
No duplicate records found.


## 7. Cross-check against the GitHub table
Validation of the parser: rows are matched on Type, Airline, Year and Month and the passenger counts compared. The GitHub copy parses the same DGCA files, so agreement checks our parser, not DGCA's figures.

In [13]:
gh=pd.read_csv(RAW/"github"/gh_files["carrier"][1])
gh=gh[gh["Year"]>=START_YEAR].copy()
for d in [dgca,gh]:
    d["Type"]=d["Type"].astype("string").str.strip()
    d["Airline"]=d["Airline"].astype("string").str.strip()
    d["Year"]=pd.to_numeric(d["Year"],errors="coerce")
    d["Month"]=pd.to_numeric(d["Month"],errors="coerce")
    d["Passenger Number"]=pd.to_numeric(d["Passenger Number"].astype("string").str.replace(",","",regex=False),errors="coerce")
    assert d[KEYS].notna().all().all()
    d["Year"]=d["Year"].astype(int)
    d["Month"]=d["Month"].astype(int)

dgca_dupes=dgca[dgca.duplicated(KEYS,keep=False)]
gh_dupes=gh[gh.duplicated(KEYS,keep=False)]
print("DGCA duplicate keys:",len(dgca_dupes),"| GitHub duplicate keys:",len(gh_dupes))
if not dgca_dupes.empty or not gh_dupes.empty:
    raise ValueError("Resolve duplicate keys before performing the cross-source merge.")

m=dgca.merge(gh,on=KEYS,how="outer",suffixes=("_dgca","_gh"),indicator=True,validate="one_to_one")
print("\nRecord match status:\n",m["_merge"].value_counts().to_string())
both=m[m["_merge"]=="both"].copy()
a=both["Passenger Number_dgca"].to_numpy(dtype=float)
b=both["Passenger Number_gh"].to_numpy(dtype=float)
both["passenger_count_equal"]=np.isclose(a,b,rtol=0,atol=1e-6,equal_nan=True)
print(f"\nPassenger counts equal: {both.passenger_count_equal.sum()}/{len(both)} ({both.passenger_count_equal.mean():.1%})" if len(both) else "\nNo matched rows to compare.")
print("\nOnly in DGCA parse:\n",m[m["_merge"]=="left_only"].groupby(["Airline","Year"]).size().head(15).to_string())
print("\nOnly in GitHub:\n",m[m["_merge"]=="right_only"].groupby(["Airline","Year"]).size().head(15).to_string())
m.to_csv(INTERIM/f"cross_source_comparison_{STAMP}.csv",index=False)
display(both.loc[~both.passenger_count_equal,KEYS+["Passenger Number_dgca","Passenger Number_gh"]].head(20))
n_equal,n_both=int(both.passenger_count_equal.sum()),len(both)

DGCA duplicate keys: 0 | GitHub duplicate keys: 0

Record match status:
 _merge
both          2966
left_only     2878
right_only     533

Passenger counts equal: 2966/2966 (100.0%)

Only in DGCA parse:
 Airline       Year
AIX Connect   2024    31
Air Carnival  2017     8
Air Costa     2017    10
Air Deccan    2017    11
              2020    47
              2021    48
Air Heritage  2019    42
              2020    45
              2021    48
Air India     2019    12
              2020    13
              2021    12
              2022    17
              2023    18
              2024    10

Only in GitHub:
 Airline             Year
Blue Dart Aviation  2017    12
                    2018    12
                    2019    16
                    2020    31
                    2021    34
                    2022    35
                    2023    24
                    2024    29
                    2025    24
                    2026    14
QuikJet Cargo       2017    24
                   

,Type,Airline,Year,Month,Passenger Number_dgca,Passenger Number_gh


## 8. Save the interim table
Cleaning is a separate step that reads this file, so the raw and interim stages stay reproducible.

**Format.** Parquet is the working format because it keeps column types, including missing numbers and dates, is compact, and needs no re-parsing. A CSV copy is saved so the table can be opened and inspected in any tool. SQLite would add nothing here: the analysis reads whole tables and needs no joins or queries inside a database. Dtypes are checked after reading the file back.

In [14]:
dgca=dgca.sort_values(KEYS+["Source File","Source Sheet"]).reset_index(drop=True)
csv_path=INTERIM/f"dgca_domestic_airline_monthly_{STAMP}.csv"
parquet_path=INTERIM/f"dgca_domestic_airline_monthly_{STAMP}.parquet"
log_path=INTERIM/f"parse_log_{STAMP}.csv"
dgca.to_csv(csv_path,index=False)
dgca.to_parquet(parquet_path,index=False)
parse_log.to_csv(log_path,index=False)
check=pd.read_parquet(parquet_path)
pd.testing.assert_frame_equal(dgca,check,check_dtype=False)
dtype_check=pd.DataFrame({"written":dgca.dtypes.astype(str),"read back":check.dtypes.astype(str)})
dtype_check["same"]=dtype_check.written==dtype_check["read back"]
print("dtypes after reading the Parquet file back:");display(dtype_check)
variants=dgca.assign(file_stem=dgca["Source File"].str.replace(r"\s*\d+\.xlsx?$","",regex=True,case=False).str.strip()).groupby(["file_stem","Airline"]).size().rename("rows").reset_index()
variants.to_csv(INTERIM/f"airline_name_variants_{STAMP}.csv",index=False)
month_variants=pd.DataFrame([(k[0],k[1],v) for k,v in sorted(month_labels.items())],columns=["raw_label","month","rows"])
month_variants.to_csv(INTERIM/f"month_label_variants_{STAMP}.csv",index=False)
print(variants.file_stem.nunique(),"airline spellings in file names ->",variants.Airline.nunique(),"names |",len(month_variants),"month spellings -> 12 months")
print("CSV saved:",csv_path)
print("Parquet saved and read-back validated:",parquet_path)
print("Parse log saved:",log_path)
print("Final shape:",dgca.shape)
print("Unique airlines:",dgca.Airline.nunique())
print("Years:",dgca.Year.min(),"-",dgca.Year.max())
print("Traffic types:",dgca.Type.unique().tolist())

dtypes after reading the Parquet file back:


,written,read back,same
Type,string,string,True
Airline,string,string,True
Year,int64,int64,True
Month,int64,int64,True
Source File,str,str,True
Source Sheet,str,str,True
Aircraft Number,Float64,Float64,True
Aircraft Hours,Float64,Float64,True
Aircraft Kilometres,Float64,Float64,True
Passenger Number,Float64,Float64,True


42 airline spellings in file names -> 27 names | 14 month spellings -> 12 months
CSV saved: /Users/pranauvmac/Programming/Acads_Python/DA2402_DCV/DA2402-Project/data/interim/dgca_domestic_airline_monthly_2026-11-01.csv
Parquet saved and read-back validated: /Users/pranauvmac/Programming/Acads_Python/DA2402_DCV/DA2402-Project/data/interim/dgca_domestic_airline_monthly_2026-11-01.parquet
Parse log saved: /Users/pranauvmac/Programming/Acads_Python/DA2402_DCV/DA2402-Project/data/interim/parse_log_2026-11-01.csv
Final shape: (5844, 22)
Unique airlines: 27
Years: 2017 - 2026
Traffic types: ['NonScheduledDomestic', 'NonScheduledInternational', 'ScheduledDomestic', 'ScheduledInternational']


In [15]:
metadata={"collection_date":STAMP,"start_year":START_YEAR,"include_citypair":True,"excluded_names_pattern":EXCLUDE,"github_repo":"Vonter/india-aviation-traffic","github_commit":sha,"files_in_manifest":len(manifest),"downloaded":int((manifest.status=="downloaded").sum()),"cached":int((manifest.status=="cached").sum()),"failed":int((~manifest.status.isin(["downloaded","cached"])).sum()),"parsed_rows":len(dgca),"parsed_files":int((parse_log.status=="parsed").sum()),"files_with_parse_errors":int((parse_log.status=="error").sum()),"parquet_readback_validated":True}
with open(INTERIM/f"collection_metadata_{STAMP}.json","w") as f:
    json.dump(metadata,f,indent=2)
print(json.dumps(metadata,indent=2))

{
  "collection_date": "2026-11-01",
  "start_year": 2017,
  "include_citypair": true,
  "excluded_names_pattern": "totalint|quikjet|bluedart",
  "github_repo": "Vonter/india-aviation-traffic",
  "github_commit": "e197170f131e43169620e4f8aa520350d4cbcba9",
  "files_in_manifest": 150,
  "downloaded": 142,
  "cached": 0,
  "failed": 8,
  "parsed_rows": 5844,
  "parsed_files": 142,
  "files_with_parse_errors": 0,
  "parquet_readback_validated": true
}


## 9. City-pair passengers
The dashboard's route, city and network charts use the Vonter city-pair table and nothing else, so the interim table holds exactly those rows.

Vonter has no rows for **May 2021 and May 2024**. These two months are left missing, as they are in the dashboard. DGCA's own files for them are kept in `data/raw/dgca_citypair/` for reference, but no final output reads them.

In [16]:
cols = ["Year", "Month", "City1", "City2", "PaxToCity2", "PaxFromCity2"]
citypair = pd.read_csv(RAW / "github" / gh_files["city"][1], usecols=cols).assign(Source="vonter")
citypair = citypair.sort_values(["Year", "Month", "City1", "City2"]).reset_index(drop=True)
cp_date = pd.to_datetime(dict(year=citypair.Year, month=citypair.Month, day=1))
cp_missing = pd.date_range(cp_date.min(), cp_date.max(), freq="MS").difference(cp_date.unique())
citypair.to_csv(INTERIM / f"dgca_domestic_citypair_monthly_{STAMP}.csv", index=False)
print("saved", f"dgca_domestic_citypair_monthly_{STAMP}.csv", "| rows:", len(citypair), "| months:", cp_date.nunique(), f"| {cp_date.min():%b %Y} to {cp_date.max():%b %Y}")
print("months with no city-pair rows:", [f"{d:%b %Y}" for d in cp_missing])

al = dgca[(dgca.Type == "ScheduledDomestic") & ~dgca.Airline.str.lower().str.contains("total")]
al_tot = al.assign(Date=pd.to_datetime(dict(year=al.Year, month=al.Month, day=1))).groupby("Date")["Passenger Number"].sum()
cp_tot = (citypair.PaxToCity2 + citypair.PaxFromCity2).groupby(cp_date).sum()
ratio = (cp_tot / al_tot).dropna().astype(float).loc["2019-01-01":]
print(f"city-pair total / airline total, 2019 onward: min {ratio.min():.3f}, max {ratio.max():.3f} (should be about 1)")

saved dgca_domestic_citypair_monthly_2026-11-01.csv | rows: 66453 | months: 134 | Apr 2015 to Jul 2026
months with no city-pair rows: ['May 2021', 'May 2024']
city-pair total / airline total, 2019 onward: min 0.994, max 1.000 (should be about 1)


## 10. Coverage: what was not collected, and what the unmatched records are
The eight files that could not be downloaded are listed with what each gap means. A coverage matrix shows, for every airline and year, how many months have numbers, so a missing month is never mistaken for zero. Unmatched records in the cross-check are classified instead of being left as a count.

In [17]:
# Source files that could not be downloaded, and what each gap means for the analysis
failed = manifest[~manifest.status.isin(["downloaded", "cached"])].copy()
STEM = {"airasia": "AirAsia India", "go air": "Go First", "goair": "Go First", "trujet": "TruJet", "vistara": "Vistara"}
def airline_year(name):
    stem = re.sub(r"\.xlsx?$", "", name, flags=re.I).strip(); m = re.search(r"(\d{2})$", stem)
    key = re.sub(r"\d+$", "", stem).strip().lower()
    return STEM.get(key, key), (2000 + int(m.group(1)) if m else None)
failed[["airline", "year"]] = [airline_year(n) for n in failed.original_name]
GAP_NOTE = {"Vistara": "merged into Air India in November 2024, so no 2025 or 2026 file is expected",
            "Go First": "stopped flying in May 2023 (insolvency filing), so no 2025 or 2026 file is expected",
            "TruJet": "UNRESOLVED: no traffic in the files that did download after Feb 2022, and the 2024-26 files are not served; treat as stopped only with a source",
            "AirAsia India": "UNRESOLVED: the 2017 file is not served, so Air India Group's 2017 total is understated (outside the 2019 onward analysis)"}
failed["meaning"] = failed.airline.map(GAP_NOTE).fillna("unclassified: check")
failed[["original_name", "airline", "year", "status", "meaning", "url"]].to_csv(INTERIM / f"download_failures_{STAMP}.csv", index=False)
print(f"{len(failed)} files could not be downloaded; every one answers HTTP 403, and the capitalisation variants tried for them were not served either.")
display(failed[["original_name", "airline", "year", "status", "meaning"]])

8 files could not be downloaded; every one answers HTTP 403, and the capitalisation variants tried for them were not served either.


,original_name,airline,year,status,meaning
31,airasia17.xlsx,AirAsia India,2017,http_403,"UNRESOLVED: the 2017 file is not served, so Air India Group's 2017..."
74,go air25.xlsx,Go First,2025,http_403,"stopped flying in May 2023 (insolvency filing), so no 2025 or 2026..."
75,go air26.xlsx,Go First,2026,http_403,"stopped flying in May 2023 (insolvency filing), so no 2025 or 2026..."
135,trujet24.xlsx,TruJet,2024,http_403,UNRESOLVED: no traffic in the files that did download after Feb 20...
136,trujet25.xlsx,TruJet,2025,http_403,UNRESOLVED: no traffic in the files that did download after Feb 20...
137,trujet26.xlsx,TruJet,2026,http_403,UNRESOLVED: no traffic in the files that did download after Feb 20...
146,vistara25.xlsx,Vistara,2025,http_403,"merged into Air India in November 2024, so no 2025 or 2026 file is..."
147,vistara26.xlsx,Vistara,2026,http_403,"merged into Air India in November 2024, so no 2025 or 2026 file is..."


In [18]:
# Coverage matrix: for each airline and year, how many months have a row with numbers (scheduled domestic). A missing month is missing, not zero.
live = dgca[dgca.Type.eq("ScheduledDomestic") & ~dgca.Airline.astype(str).str.lower().str.contains("total")].copy()
live = live[live[METRICS].apply(pd.to_numeric, errors="coerce").notna().any(axis=1)]
coverage = live.groupby(["Airline", "Year"]).Month.nunique().unstack("Year").fillna(0).astype(int)
coverage.to_csv(INTERIM / f"coverage_matrix_scheduled_domestic_{STAMP}.csv")
print("months with numbers per airline and year (12 = complete; a blank is a year with no data):")
display(coverage.replace(0, pd.NA))

# Unmatched records in the cross-source comparison: classify them instead of leaving them as a count
mm = m.copy()
dg_cols = [f"{c}_dgca" for c in METRICS if f"{c}_dgca" in mm.columns]
blank = mm[dg_cols].apply(pd.to_numeric, errors="coerce").isna().all(axis=1)
gh_last = mm.loc[mm._merge.isin(["both", "right_only"]), ["Year", "Month"]].apply(tuple, axis=1).max()       # the latest month in the GitHub copy
newer = mm[["Year", "Month"]].apply(tuple, axis=1) > gh_last
mm["match_class"] = np.select(
    [mm._merge.eq("both"),
     mm._merge.eq("left_only") & mm.Type.astype(str).str.contains("International"),
     mm._merge.eq("left_only") & blank,
     mm._merge.eq("left_only") & newer,
     mm._merge.eq("left_only"),
     mm._merge.eq("right_only") & mm.Airline.astype(str).str.lower().str.replace(r"[^a-z]", "", regex=True).str.contains(EXCLUDE, regex=True),   # "Blue Dart Aviation" -> "bluedartaviation"
     mm._merge.eq("right_only")],
    ["matched", "international section (the GitHub domestic file has none)", "empty template row (nothing to compare)", "month newer than the GitHub copy (its commit predates it)",
     "live domestic row missing from the GitHub copy: CHECK", "cargo carrier, excluded by design", "only in the GitHub copy: CHECK"], default="other")
summary = mm.groupby("match_class").size().rename("records").to_frame()
summary.to_csv(INTERIM / f"cross_source_match_classes_{STAMP}.csv")
display(summary)
agree = mm[mm._merge.eq("both")]
print(f"Passenger counts agree on {n_equal} of {n_both} matched records. "
      "This checks our parser against Vonter's parse of the same DGCA files; it does not check DGCA's own figures.")
mm[mm.match_class.str.contains("CHECK")][KEYS + ["Source File", "match_class"]].to_csv(INTERIM / f"cross_source_to_check_{STAMP}.csv", index=False)

months with numbers per airline and year (12 = complete; a blank is a year with no data):


Year,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Airline,,,,,,,,,,
AIX Connect,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,9,<NA>,<NA>
Air Carnival,4,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Air Costa,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Air Deccan,1,12,<NA>,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Air Heritage,<NA>,<NA>,6,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Air India,12,12,12,11,12,12,12,12,12,9
Air India Express,12,12,12,11,6,6,4,12,12,8
Air Odisha,<NA>,11,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
Air Taxi,<NA>,<NA>,<NA>,<NA>,8,<NA>,<NA>,<NA>,<NA>,<NA>


,records
match_class,
"cargo carrier, excluded by design",533
empty template row (nothing to compare),1037
international section (the GitHub domestic file has none),1797
live domestic row missing from the GitHub copy: CHECK,9
matched,2966
month newer than the GitHub copy (its commit predates it),35


Passenger counts agree on 2966 of 2966 matched records. This checks our parser against Vonter's parse of the same DGCA files; it does not check DGCA's own figures.
